In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("bank.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.columns:
    print(df[col].value_counts(dropna=False))

In [ ]:
df[["default","housing","loan","deposit"]]=(df[["default","housing","loan","deposit"]]=="yes").astype(int)

In [ ]:
df

In [ ]:
#for col in df.columns:
 #   print(df[col].value_counts()) 

In [ ]:
df=pd.get_dummies(df,columns=["job","marital","education","contact","month","poutcome"],dtype=int)

In [ ]:
df

In [ ]:
# Train-test split and model selection
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV,cross_val_score,StratifiedKFold


# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Model evaluation metrics
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,roc_auc_score,roc_curve


# Pipeline
from sklearn.pipeline import Pipeline

# Feature importance
from sklearn.inspection import permutation_importance

# For plotting decision trees in the forest
from sklearn.tree import plot_tree

In [ ]:
#there is data leakage,for betterment of model dro

In [ ]:
X = df.drop(["deposit"], axis=1)
y = df["deposit"]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
pipe=Pipeline([("model",RandomForestClassifier(random_state=42))])

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt"]
}
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
grid=GridSearchCV(pipe,param_grid,scoring=scoring,refit="accuracy",n_jobs=-1,cv=skf)
grid.fit(X_train,y_train)

In [ ]:
y_pred=grid.predict(X_test)

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
precision_score(y_test,y_pred)

In [ ]:
f1_score(y_test,y_pred)

In [ ]:
recall_score(y_test,y_pred)

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
pd.DataFrame(classification_report(y_test,y_pred,output_dict=True))

In [ ]:
roc_auc_score(y_test,y_pred)

In [ ]:
roc_curve(y_test,y_pred)

In [ ]:
accuracy_score(y_train,grid.predict(X_train))

In [ ]:
accuracy_score(y_test,grid.predict(X_test))

In [ ]:
best_model = grid.best_estimator_.named_steps["model"]

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance_df.head(20))

In [ ]:
y_proba=grid.predict_proba(X_test)[:,1]

In [ ]:
roc_auc_score(y_test,y_proba)

In [ ]:
y_proba = grid.predict_proba(X_test)[:, 1]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.plot(fpr, tpr, label="ROC Curve")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Model")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

In [ ]:
plt.plot(thresholds,tpr,label="tpr")
plt.plot(thresholds,fpr,label="fpr")
plt.plot(thresholds,tpr-fpr,label="(tpr-fpr)")
plt.xlabel("threshold")
plt.legend()

In [ ]:
j=tpr-fpr
j

In [ ]:
best_index=j.argmax()
best_threshold=thresholds[best_index]
print(best_threshold)

In [ ]:
j[best_index]

In [ ]:
tpr[best_index]

In [ ]:
fpr[best_index]

In [ ]:
y_pred=(y_proba>=best_threshold).astype(int)

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
precision_score(y_test,y_pred)

In [ ]:
recall_score(y_test,y_pred)

In [ ]:
f1_score(y_test,y_pred)

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
pd.DataFrame(classification_report(y_test,y_pred,output_dict=True))

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])

best_idx = f1_scores.argmax()

best_threshold = thresholds[best_idx]

print(best_threshold)

In [ ]:
plt.plot(thresholds,f1_scores,label="f1_scores")
plt.plot(thresholds,precision[:-1],label="precision")
plt.plot(thresholds,recall[:-1],label="recall")
plt.legend()
plt.xlabel(thresholds)
plt.show()

In [ ]:
f1_scores[best_idx]

In [ ]:
y_pred1=(y_proba>=best_threshold).astype(int)

In [ ]:
accuracy_score(y_test,y_pred1)

In [ ]:
precision_score(y_test,y_pred1)

In [ ]:
recall_score(y_test,y_pred1)

In [ ]:
f1_score(y_test,y_pred1)

In [ ]:
confusion_matrix(y_test,y_pred1)

In [ ]:
pd.DataFrame(classification_report(y_test,y_pred1,output_dict=True))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(y_test, y_proba)

In [ ]:
from sklearn.utils import all_estimators

models = all_estimators()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    grid,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)